## Silver - dim_cliente

### Transformações para a camada silver

####Importando bibliotecas

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import trim, col
from pyspark.sql.types import StringType

### dim_cliente
####Consultando dados da tabela bronze

In [0]:
df_cliente = spark.read.table("dev_credit_risk.bronze.dim_cliente_raw_s3")

#### Início das transformações
#### Retirando os espaços em branco das colunas de tipo string.

In [0]:
for field in df_cliente.schema.fields:
    if isinstance(field.dataType, StringType):
        df_cliente = df_cliente.withColumn(field.name, trim(col(field.name)))

In [0]:
df_cliente.display()

#### Normalizando os valores

In [0]:
df_cliente = (
    df_cliente
    .withColumn(
        "sexo",
        F.when(F.upper(F.col("sexo")).startswith("M"), "Masculino")
         .when(F.upper(F.col("sexo")).startswith("F"), "Feminino")
        .otherwise(F.upper(F.col("sexo")))
    )
    .withColumn(
        "estadoCivil",
        F.when(F.upper(F.col("estadoCivil")).startswith("S"), "Solteiro")
         .when(F.upper(F.col("estadoCivil")).startswith("C"), "Casado")
        .otherwise(F.upper(F.col("estadoCivil")))
    )
    .withColumn(
        "uf",
        F.when(F.upper(F.col("UF")).startswith("S"), "SP")
         .when(F.upper(F.col("UF")).startswith("R"), "RJ")
         .when(F.upper(F.col("UF")).startswith("P"), "PE")
        .otherwise(F.upper(F.col("UF")))
    )
)

####Checando o domínio. Valores únicos após normalização

In [0]:
display(df_cliente.select("sexo","estadoCivil","uf").distinct())

###Padronizando os nomes das colunas

In [0]:
df_cliente.display()

In [0]:
rename_map = {
    "idCliente": "id_cliente",  
    "nomeCliente": "nome_cliente",
    "idade": "idade",
    "sexo": "sexo",
    "estadoCivil":"estado_civil",
    "FaixaRenda": "faixa_renda",
    "uf": "uf"
}

####Renomeando as colunas

In [0]:
for old_name, new_name in rename_map.items():
    df_cliente = df_cliente.withColumnRenamed(old_name, new_name)

In [0]:
df_cliente = df_cliente.select(
    "id_cliente",
    "nome_cliente",
    "idade",
    "sexo",
    "estado_civil",
    "faixa_renda",
    "uf"
)

####Exibindo o dataframe pronto para ser inserido na tabela delta silver

####Inserindo os dados na tabela silver

In [0]:
(df_cliente.write
  .mode("overwrite")
  .format("delta")
  .saveAsTable("dev_credit_risk.silver.dim_cliente"))